# Test polytopes' volume computation — CNN

Analogous to `test_volumes.ipynb` but for `FashionCNN_Small`.

Internally calls `_prep_lp` which:
- **prunes zero-norm rows** (trivial constraints from saturated MaxPool neurons)
- **adds ε = 1e-6 slack** to absorb floating-point violations at x₀

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch

from src.models.networks import FashionCNN_Small
from src.quantization.quantize import quantize_model
from src.optim.build_polytopes_cnn import build_two_class_cnn_polytopes, \
                                          build_cnn_all_polytopes
from src.optim.compute_volumes import estimate_polytope_width

Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [2]:
torch.manual_seed(42)

## CNN (FashionMNIST)

In [3]:
# -------------------- #
# Load models and data #
# -------------------- #

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("Device:", device)

# Full-precision CNN
model = FashionCNN_Small()
model.load_state_dict(
    torch.load(os.path.join(ROOT, "checkpoints", "fashion_cnn_best.pth"),
               weights_only=True, map_location=device)
)
model.to(device).eval()

# Quantized model (bits=4 for the main test)
bits = 4
qmodel = quantize_model(model, bits=bits)
qmodel.to(device).eval()

# Dataset (correctly-classified samples for the CNN)
dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_cnn.pt",
    weights_only=False
)

x_0, c = dataset[13]
print(f"x_0 range: [{x_0.min().item():.2f}, {x_0.max().item():.2f}]")  # check bounds
x_0  = x_0.to(device)          # (1, 28, 28)
x_in = x_0.unsqueeze(0)        # (1, 1, 28, 28) — batch dim for the CNN
print(f"x_in shape: {x_in.shape}  |  label: {c}")
print("Models and dataset have been loaded.")

Device: mps
x_0 range: [-1.00, 0.99]
x_in shape: torch.Size([1, 1, 28, 28])  |  label: 5
Models and dataset have been loaded.


## Correct polytope — LP timing

Build the correct polytope (model + cls + qmodel) and estimate its mean width with a small number of directions to measure per-LP cost.

In [4]:
%%time

print("*** Correct polytope (model-only) ***\n")

# Keep small — each LP can take several minutes for a CNN.
# Increase to 50–200 on Jean Zay.
nb_directions = 2

# bounds apply to all 784 input variables
bounds = [(-1., 1.)]

A_correct, b_correct, _, _ = build_two_class_cnn_polytopes(model, qmodel, x_in, int(c))
print(f"A_correct shape: {tuple(A_correct.shape)}")

results_correct = estimate_polytope_width(
    A_correct, b_correct, bounds,
    n_directions=nb_directions,
    verbose=True,
)
results_correct

*** Correct polytope (model-only) ***

A_correct shape: (41361, 784)
Direction 0: w_correct=0.5615
Direction 1: w_correct=0.5832
CPU times: user 1min 23s, sys: 1.32 s, total: 1min 25s
Wall time: 1min 25s


{'mean_width_correct': 0.5723444487039511,
 'std_width_correct': 0.01088023199510002,
 'n_directions_used': 2}

## b-approximated polytope — paired LP (same directions)

Estimate mean width of `A_both` (model + cls + qmodel + qcls) using the **same** random directions as above (paired mode).  This gives the error estimate:

In [5]:
%%time

print(f"*** b-approximated polytope (bits={bits}) — paired estimation ***\n")

# build_cnn_all_polytopes reuses model constraints internally (computed once).
# We pass only the one qmodel we care about here.
_, _, polytopes_dict = build_cnn_all_polytopes(model, {bits: qmodel}, x_in, int(c))
A_both, b_both = polytopes_dict[bits]
print(f"A_both shape: {tuple(A_both.shape)}")

results_both = estimate_polytope_width(
    A_correct, b_correct, bounds,
    A_both=A_both, b_both=b_both,
    n_directions=nb_directions,
    verbose=True,
)
results_both

*** b-approximated polytope (bits=4) — paired estimation ***

A_both shape: (41370, 784)
Direction 0: w_correct=0.5666, w_both=0.5666
Direction 1: w_correct=0.5500, w_both=0.5500
CPU times: user 2min 39s, sys: 2.16 s, total: 2min 41s
Wall time: 2min 42s


{'mean_width_correct': 0.5582880142524748,
 'std_width_correct': 0.008320208871948065,
 'n_directions_used': 2,
 'mean_width_both': 0.5582880142533738,
 'std_width_both': 0.008320208872173884,
 'error': -1.610489519521252e-12}

## Sanity check — model and qmodel predictions

In [6]:
with torch.no_grad():
    pred_model  = torch.argmax(model(x_in)).item()
    pred_qmodel = torch.argmax(qmodel(x_in)).item()
print(f"model  prediction : {pred_model}")
print(f"qmodel prediction : {pred_qmodel}")
print(f"true label        : {int(c)}")

model  prediction : 5
qmodel prediction : 5
true label        : 5
